# Pré-processamento do dataset médico PT-BR — v3

In [1]:
!pip install -q "transformers==5.14.1" "datasets==5.0.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 58.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 71.0 MB/s eta 0:00:00


## 1. Configuração

`SEED` tem que ser idêntico ao do notebook de fine-tuning.

Os limites de tamanho são as alavancas: mexer neles muda acurácia **e** tempo de
treino ao mesmo tempo. Resposta longa é resposta imprevisível e cara.

In [ ]:
import os, re, json, math, random, hashlib, unicodedata
from collections import defaultdict, Counter

import numpy as np
from datasets import Dataset, load_dataset
from transformers import AutoTokenizer

SEED = 42
random.seed(SEED); np.random.seed(SEED)

BASE_DRIVE = "/content/drive/MyDrive/Colab Notebooks/FIAP - Tech 3"

# Entrada: o dataset bruto curado (mesmo arquivo da versão anterior)
RAW_DATASET_JSONL = f"{BASE_DRIVE}/dataset_medpt_curado.jsonl"

# Saída: aponte o PREPROCESSED_DIR do notebook de fine-tuning para cá
OUTPUT_DIR = f"{BASE_DRIVE}/dataset_preprocessado_v3"

# Tokenizer usado só para CONTAR tokens. Tem que ser o mesmo do treino.
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

# ---------------------------------------------------------------- limites
MIN_TOK_RESPOSTA = 40
MAX_TOK_RESPOSTA = 200  
MAX_TOK_PERGUNTA = 160  
MIN_OVERLAP      = 0.06 
LIMIAR_NEAR_DUP  = 0.88 
EXCLUIR_ESPECIALIDADES = ("homeopat", "acupuntur", "terapia complementar",
                          "terapeuta complementar", "floral", "reiki")

SPLIT = (0.80, 0.10, 0.10)

SYSTEM_PROMPT = (
    "Você é um assistente médico de apoio. Responda em português, "
    "de forma objetiva e baseada em evidências, e não substitua "
    "a avaliação de um profissional de saúde."
)
SYSTEM_PROMPT_HASH = hashlib.sha256(SYSTEM_PROMPT.encode()).hexdigest()[:12]

print("SYSTEM_PROMPT hash:", SYSTEM_PROMPT_HASH)
print(SYSTEM_PROMPT)

SYSTEM_PROMPT hash: e7f03852bc2c
Você é um assistente médico de apoio. Responda em português, de forma objetiva e baseada em evidências, e não substitua a avaliação de um profissional de saúde.


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Carga do bruto e detecção de schema

Os nomes das colunas do dataset bruto variam. A célula detecta os campos de
pergunta e resposta em vez de assumir.

In [4]:
bruto = load_dataset("json", data_files=RAW_DATASET_JSONL, split="train")
print(f"{len(bruto)} registros brutos")
print("colunas:", bruto.column_names)

def _achar(cands, colunas):
    for c in cands:
        if c in colunas:
            return c
    return None

COL_PERGUNTA = _achar(["question", "pergunta", "entrada", "input"], bruto.column_names)
COL_RESPOSTA = _achar(["answer", "resposta", "saida", "output"], bruto.column_names)
COL_ESPEC    = _achar(["medical_specialty", "especialidade"], bruto.column_names)
COL_COND     = _achar(["condition", "condicao"], bruto.column_names)
COL_TIPO     = _achar(["question_type", "tipo_pergunta"], bruto.column_names)
COL_ID       = _achar(["id_original", "id"], bruto.column_names)

assert COL_PERGUNTA and COL_RESPOSTA, \
    f"Não achei pergunta/resposta em {bruto.column_names}. Ajuste manualmente."
print(f"pergunta='{COL_PERGUNTA}' | resposta='{COL_RESPOSTA}' | espec='{COL_ESPEC}'")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
n_tok = lambda t: len(tokenizer(t, add_special_tokens=False)["input_ids"])

Generating train split: 0 examples [00:00, ? examples/s]

10000 registros brutos
colunas: ['id', 'question', 'answer', 'condition', 'medical_specialty', 'question_type']
pergunta='question' | resposta='answer' | espec='medical_specialty'


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

## 3. Normalização

Sem LLM, sem API. Remove saudação, despedida, caixa alta enfática e espaço extra.

Essas são as posições de **maior entropia** do alvo: "Olá", "Oi", "Bom dia",
"Boa sorte", "Estou à disposição" — mil variações para dizer a mesma coisa, e cada
variação é um token que o modelo nunca vai acertar. Tirar isso sobe a acurácia
e corta tokens ao mesmo tempo.

In [ ]:
SAUDACOES = re.compile(
    r"^\s*(ol[áa]|oi|bom dia|boa tarde|boa noite|prezad[oa]s?|car[oa]s?)"
    r"[\s,!.:;–-]*", re.I)

DESPEDIDAS = re.compile(
    r"(estou [àa] disposi[çc][ãa]o|fico [àa] disposi[çc][ãa]o|boa sorte|melhoras|"
    r"abra[çc]os?|att\.?|atenciosamente|espero ter ajudado|"
    r"qualquer d[úu]vida[^.]*)[\s,!.:;–-]*$", re.I)

CONTATO = re.compile(
    r"(https?://|www\.|@[\w.]+|\(?\d{2}\)?\s?9?\d{4}[-\s]?\d{4}|"
    r"\bCRM[\s-]?\w*\s*\d+|consult[óo]rio|agende (sua|uma) (consulta|avalia))", re.I)

CAPS_RUN = re.compile(
    r"\b[A-ZÁÀÂÃÉÊÍÓÔÕÚÜÇ]{2,}(?:[\s,;:!?…-]+[A-ZÁÀÂÃÉÊÍÓÔÕÚÜÇ]+)*\b")

def _abaixar_caps(m):
    trecho = m.group(0)
    palavras = re.findall(r"[A-ZÁÀÂÃÉÊÍÓÔÕÚÜÇ]+", trecho)
    maior = max(len(p) for p in palavras)
    if (len(palavras) >= 2 and maior >= 6) or maior >= 7:
        return trecho.lower()
    return trecho

# Checagem de nomes de profissionais (ex.: "Dr. Ricardo Ribeiro"). O título é
# case-insensitive, o nome não.
NOME_PROPRIO = re.compile(r"(?i:\bdra?\.?)\s+[A-ZÁÀÂÃÉÊÍÓÔÕÚÇ][A-Za-zÀ-ÿ]{2,}")

# resposta que é só "procure um médico" e nada mais
NAO_RESPOSTA = re.compile(
    r"^(procure|busque|consulte|agende|marque)\b[^.]{0,120}\.?\s*$", re.I)

def normalizar(t):
    t = unicodedata.normalize("NFC", str(t)).strip()
    t = SAUDACOES.sub("", t)
    for _ in range(3):                      # despedidas encadeadas
        novo = DESPEDIDAS.sub("", t).strip()
        if novo == t:
            break
        t = novo

    t = CAPS_RUN.sub(_abaixar_caps, t)
    # a frase pode ter ficado começando em minúscula depois do passo acima
    t = re.sub(r"(^|[.!?]\s+)([a-zà-ú])", lambda m: m.group(1) + m.group(2).upper(), t)
    t = re.sub(r"[ \t]+", " ", t)
    t = re.sub(r"\n{3,}", "\n\n", t)
    return t.strip()

STOP = set(
    "a o as os um uma de do da dos das em no na nos nas por para com sem sobre e ou "
    "mas que se como quando onde qual quais meu minha seu sua isso isto esse essa ao "
    "aos à às pelo pela ser estar tem ter há muito mais menos já não sim".split())

def conteudo(t):
    return {p for p in re.findall(r"[a-zà-ú]{4,}", t.lower()) if p not in STOP}

# Sanidade: grito vira minúscula, sigla clínica não é tocada.
_casos = [
    "Olá. procure um médico QUE ATENDA DEMORADAMENTE E LHE EXPLIQUE TUDO. Boa sorte",
    "Seu exame de TSH e T4 LIVRE está normal, assim como o HIV e a RM.",
]
for _c in _casos:
    print("ANTES :", _c)
    print("DEPOIS:", normalizar(_c), "\n")

ANTES : Olá. procure um médico QUE ATENDA DEMORADAMENTE E LHE EXPLIQUE TUDO. Boa sorte
DEPOIS: Procure um médico que atenda demoradamente e lhe explique tudo. 

ANTES : Seu exame de TSH e T4 LIVRE está normal, assim como o HIV e a RM.
DEPOIS: Seu exame de TSH e T4 LIVRE está normal, assim como o HIV e a RM. 



## 4. MinHash para near-duplicatas

Fórum médico repete pergunta quase idêntica o tempo todo. Se `train` e `test`
compartilham near-duplicatas, **toda métrica infla**. O dedup roda aqui, antes do
split — depois do split não adianta, o vazamento já aconteceu.

In [6]:
N_PERM, N_BANDS = 64, 16
_A = np.random.randint(1, 2**31 - 1, N_PERM, dtype=np.int64)
_B = np.random.randint(0, 2**31 - 1, N_PERM, dtype=np.int64)
_P = np.int64(2_147_483_647)

def assinatura(texto, k=5):
    palavras = re.findall(r"[a-zà-ú0-9]+", texto.lower())
    if len(palavras) < k:
        palavras += ["_"] * (k - len(palavras))
    shingles = {hash(" ".join(palavras[i:i + k])) & 0x7FFFFFFF
                for i in range(len(palavras) - k + 1)}
    if not shingles:
        return np.zeros(N_PERM, dtype=np.int64)
    h = np.fromiter(shingles, dtype=np.int64)
    return ((_A[:, None] * h[None, :] + _B[:, None]) % _P).min(axis=1)

def remover_near_dups(regs, limiar=LIMIAR_NEAR_DUP):
    sigs = [assinatura(r["entrada"] + " " + r["saida"]) for r in regs]
    por_banda, candidatos = defaultdict(list), set()
    linhas = N_PERM // N_BANDS
    for i, s in enumerate(sigs):
        for b in range(N_BANDS):
            chave = (b, s[b * linhas:(b + 1) * linhas].tobytes())
            for j in por_banda[chave]:
                candidatos.add((j, i))
            por_banda[chave].append(i)
    fora = set()
    for j, i in sorted(candidatos):
        if j in fora or i in fora:
            continue
        if float((sigs[j] == sigs[i]).mean()) >= limiar:
            fora.add(i)
    return [r for k, r in enumerate(regs) if k not in fora], len(fora)

## 5. Filtros

Ordem obrigatória: **normalizar → filtrar → dedup → split**.

Cada descarte é contado separadamente. Se a taxa total sair fora de 25–45%, os
limites da seção 1 estão frouxos ou agressivos demais.

In [7]:
contador = Counter()
mantidos = []

for ex in bruto:
    esp = str(ex.get(COL_ESPEC, "") or "").lower()
    if any(t in esp for t in EXCLUIR_ESPECIALIDADES):
        contador["escopo_terapia_complementar"] += 1; continue

    p_bruto, r_bruto = str(ex[COL_PERGUNTA]), str(ex[COL_RESPOSTA])

    # nomes: no texto bruto (ver comentário na seção 3)
    if NOME_PROPRIO.search(r_bruto) or NOME_PROPRIO.search(p_bruto):
        contador["nome_proprio_residual"] += 1; continue

    p = normalizar(p_bruto)
    r = normalizar(r_bruto)

    if not p or not r:
        contador["vazio_apos_normalizar"] += 1; continue
    if CONTATO.search(r):
        contador["contato_autopromocao"] += 1; continue
    if NAO_RESPOSTA.match(r):
        contador["nao_resposta"] += 1; continue

    nr, npg = n_tok(r), n_tok(p)
    if nr < MIN_TOK_RESPOSTA:
        contador["resposta_curta"] += 1; continue
    if nr > MAX_TOK_RESPOSTA:
        contador["resposta_longa"] += 1; continue
    if npg > MAX_TOK_PERGUNTA:
        contador["pergunta_longa"] += 1; continue

    cp, cr = conteudo(p), conteudo(r)
    if not cp or len(cp & cr) / len(cp) < MIN_OVERLAP:
        contador["ignora_a_pergunta"] += 1; continue

    mantidos.append({
        "instrucao": SYSTEM_PROMPT,      # <- o MESMO para todas as linhas
        "entrada": p,
        "saida": r,
        "n_tok_pergunta": npg,
        "n_tok_resposta": nr,
        "id_original": ex.get(COL_ID, None),
        "condition": ex.get(COL_COND, None),
        "medical_specialty": ex.get(COL_ESPEC, None),
        "question_type": ex.get(COL_TIPO, None),
    })

print(f"filtros: {len(bruto)} -> {len(mantidos)}")
mantidos, n_dup = remover_near_dups(mantidos)
contador["near_duplicatas"] = n_dup
print(f"dedup:   -> {len(mantidos)}  ({n_dup} near-dups removidas)")

taxa = 1 - len(mantidos) / len(bruto)
print(f"\ntaxa de descarte: {taxa:.1%}")
for k, v in contador.most_common():
    print(f"  {k:32s} {v:5d}  ({v/len(bruto):5.1%})")

if not 0.25 <= taxa <= 0.45:
    print("\nAVISO: taxa fora de 25-45%. Revise os limites da seção 1.")

filtros: 10000 -> 4871
dedup:   -> 4870  (1 near-dups removidas)

taxa de descarte: 51.3%
  ignora_a_pergunta                 1833  (18.3%)
  resposta_curta                    1467  (14.7%)
  escopo_terapia_complementar        718  ( 7.2%)
  resposta_longa                     665  ( 6.7%)
  nome_proprio_residual              149  ( 1.5%)
  contato_autopromocao               126  ( 1.3%)
  pergunta_longa                     108  ( 1.1%)
  nao_resposta                        63  ( 0.6%)
  near_duplicatas                      1  ( 0.0%)

AVISO: taxa fora de 25-45%. Revise os limites da seção 1.


## 6. `max_length_sugerido`

A versão anterior usava o p95 (424) e **truncava 5% dos exemplos no meio da
resposta** — o que ensina ao modelo que resposta pode acabar sem EOS, e isso vaza
direto para a geração.

Como a seção 5 já limitou resposta e pergunta por cima, aqui dá para usar o
**máximo real** do conjunto: 0% truncado, e ainda assim um valor bem menor que 424.

In [8]:
OVERHEAD_CHAT = n_tok(SYSTEM_PROMPT) + 24     # tags <|im_start|> etc.

comprimentos = [r["n_tok_pergunta"] + r["n_tok_resposta"] + OVERHEAD_CHAT
                for r in mantidos]

MAX_LENGTH_SUGERIDO = int(math.ceil(max(comprimentos) / 8) * 8)   # múltiplo de 8

media_total = float(np.mean(comprimentos))
media_resp  = float(np.mean([r["n_tok_resposta"] for r in mantidos]))

print(f"max_length_sugerido : {MAX_LENGTH_SUGERIDO}   (a versão anterior usava 424)")
print(f"média de tokens     : {media_total:.1f}   (antes: 276,8)")
print(f"média da resposta   : {media_resp:.1f}")
print(f"% com gradiente     : {100*media_resp/media_total:.1f}%   (antes: 32%)")
print(f"truncados           : 0.00%  (por construção)")

max_length_sugerido : 408   (a versão anterior usava 424)
média de tokens     : 212.9   (antes: 276,8)
média da resposta   : 99.4
% com gradiente     : 46.7%   (antes: 32%)
truncados           : 0.00%  (por construção)


## 7. Split, verificação de vazamento e escrita

Split 80/10/10 com a mesma `SEED` do treino. A coluna `n_tokens` vai junto: o
notebook de fine-tuning usa ela para agrupar batches por tamanho e economizar
padding.

In [9]:
random.Random(SEED).shuffle(mantidos)
n = len(mantidos)
c1, c2 = int(n * SPLIT[0]), int(n * (SPLIT[0] + SPLIT[1]))
splits = {"train": mantidos[:c1], "validation": mantidos[c1:c2], "test": mantidos[c2:]}

# vazamento train <-> test
sig_train = {assinatura(r["entrada"] + " " + r["saida"]).tobytes() for r in splits["train"]}
vazou = sum(assinatura(r["entrada"] + " " + r["saida"]).tobytes() in sig_train
            for r in splits["test"])
print("near-duplicates entre train e test:", vazou)

os.makedirs(OUTPUT_DIR, exist_ok=True)
for nome, regs in splits.items():
    Dataset.from_list([{
        "instrucao": r["instrucao"],
        "entrada": r["entrada"],
        "saida": r["saida"],
        "n_tokens": r["n_tok_pergunta"] + r["n_tok_resposta"] + OVERHEAD_CHAT,
        "id_original": r["id_original"],
        "condition": r["condition"],
        "medical_specialty": r["medical_specialty"],
        "question_type": r["question_type"],
    } for r in regs]).to_json(f"{OUTPUT_DIR}/{nome}.jsonl", force_ascii=False)
    print(f"{nome:11s} {len(regs):5d}")

metadata = {
    "versao": "v3",
    "max_length_sugerido": MAX_LENGTH_SUGERIDO,
    "system_prompt": SYSTEM_PROMPT,
    "system_prompt_hash": SYSTEM_PROMPT_HASH,
    "seed": SEED,
    "model_id_tokenizer": MODEL_ID,
    "limpeza": {
        "total_bruto": len(bruto),
        "mantido": n,
        "taxa_descarte": round(taxa, 4),
        "por_filtro": dict(contador.most_common()),
    },
    "splits": {k: len(v) for k, v in splits.items()},
    "near_duplicates_train_test": int(vazou),
    "tokens": {
        "media_total": round(media_total, 1),
        "media_resposta": round(media_resp, 1),
        "pct_com_gradiente": round(100 * media_resp / media_total, 1),
        "max_total": int(max(comprimentos)),
        "pct_truncados": 0.0,
    },
    "limites": {
        "MIN_TOK_RESPOSTA": MIN_TOK_RESPOSTA, "MAX_TOK_RESPOSTA": MAX_TOK_RESPOSTA,
        "MAX_TOK_PERGUNTA": MAX_TOK_PERGUNTA, "MIN_OVERLAP": MIN_OVERLAP,
        "LIMIAR_NEAR_DUP": LIMIAR_NEAR_DUP,
    },
}
with open(f"{OUTPUT_DIR}/preprocessing_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("\n" + json.dumps(metadata, ensure_ascii=False, indent=2))
assert vazou == 0, f"{vazou} near-duplicates entre train e test"
print(f"\nOK -> {OUTPUT_DIR}")
print("Aponte PREPROCESSED_DIR do notebook de fine-tuning para este diretório.")

near-duplicates entre train e test: 0


Creating json from Arrow format:   0%|          | 0/4 [00:00<?, ?ba/s]

train        3896


Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

validation    487


Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

test          487

{
  "versao": "v3",
  "max_length_sugerido": 408,
  "system_prompt": "Você é um assistente médico de apoio. Responda em português, de forma objetiva e baseada em evidências, e não substitua a avaliação de um profissional de saúde.",
  "system_prompt_hash": "e7f03852bc2c",
  "seed": 42,
  "model_id_tokenizer": "Qwen/Qwen2.5-3B-Instruct",
  "limpeza": {
    "total_bruto": 10000,
    "mantido": 4870,
    "taxa_descarte": 0.513,
    "por_filtro": {
      "ignora_a_pergunta": 1833,
      "resposta_curta": 1467,
      "escopo_terapia_complementar": 718,
      "resposta_longa": 665,
      "nome_proprio_residual": 149,
      "contato_autopromocao": 126,
      "pergunta_longa": 108,
      "nao_resposta": 63,
      "near_duplicatas": 1
    }
  },
  "splits": {
    "train": 3896,
    "validation": 487,
    "test": 487
  },
  "near_duplicates_train_test": 0,
  "tokens": {
    "media_total": 212.9,
    "media_resposta": 99.4,
    "pct_com_gradiente": 46.7,
    "max_total": 408,
 

## 8. Conferência manual — 15 exemplos

Este passo é manual e não dá para pular. Procure: saudação sobrando, caixa alta,
telefone/CRM, nome de profissional, resposta que não responde a pergunta.

Se aparecer sujeira, ajuste o regex ou o limite na seção 1 e re-rode.

In [10]:
for i, r in enumerate(splits["train"][:15]):
    print("=" * 95)
    print(f"[{i+1}] ({r['n_tok_pergunta']} tok) PERGUNTA: {r['entrada'][:220]}")
    print(f"\n    ({r['n_tok_resposta']} tok) RESPOSTA: {r['saida'][:520]}\n")

[1] (65 tok) PERGUNTA: A compulsão alimentar é considerada uma doença? Pq consultei um psiquiatra e ele me disse que eu sofria de transtorno de ansiedade e não de compulsão, agora estou confusa. Posso ter os dois? Eles estão relacionados? Quai

    (130 tok) RESPOSTA: Compulsão alimentar é uma doença em que a pessoa sente uma necessidade de comer mesmo quando ela está satisfeita. A pessoa ingere grandes quantidades de comida em pouco tempo e está sim ligada a ansiedade. Neste caso a comida entra como uma forma de compensação de algum gatilho emocional ou estresse que a pessoa apresente, descontando na comida. Os dois podem ocorrer juntos. Sugiro que busque avaliação do psicólogo para melhor definir os sintomas que você apresenta e a intervenção terapêutica mais eficaz em cada ca

[2] (84 tok) PERGUNTA: Sinto minhas pernas formigando sempre que eu as dobro ou deixo em uma posição por um tempo. Sei que isso é normal, mas acontece muito mais rápido que o normal. O que pode ser? Tenho apen